# Machine Learning Aplicado ao E-commerce Olist
## Modelos Preditivos e de Classificação

**Analista:** Luís  
**Contexto:** Este notebook documenta a pesquisa e prototipagem dos modelos de Machine Learning propostos na análise exploratória consolidada (`main_analysis.ipynb`). O objetivo é transformar os padrões identificados — especialmente o impacto do atraso na satisfação — em modelos preditivos acionáveis.

---

### O Que Este Notebook Cobre

A análise exploratória do time revelou dois problemas centrais que podem ser endereçados com ML:

1. **Predição de Atraso** — saber, no momento da compra, se aquele pedido tem risco de atrasar (Classificação Binária)
2. **Predição de Prazo de Entrega** — estimar com precisão quantos dias o pedido vai levar (Regressão)

Ambos alimentam diretamente a página `11_Machine_Learning.py` do dashboard Streamlit.

---

### Dados Chave que Motivam Este Trabalho

| Descoberta da EDA | Impacto |
|---|---|
| Nota média com atraso: **2,47** vs sem atraso: **3,67** | Atraso derruba 1,2 ponto na avaliação |
| **95,5%** dos clientes que sofreram atraso não recompram | Custo permanente de LTV |
| **70,1%** das avaliações ruins são de vendas interestaduais | Rota é o maior preditor de risco |
| Top 20% vendedores = **82,3%** da receita | Risco concentrado e mensurável |

## Fase 1: Setup e Carregamento de Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os
warnings.filterwarnings('ignore')

# Configuração visual — padrão do projeto
sns.set_theme(style="whitegrid")
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['figure.facecolor'] = '#0d0c44'
plt.rcParams['axes.facecolor'] = '#0d0c44'
plt.rcParams['axes.edgecolor'] = '#3a3880'
plt.rcParams['axes.labelcolor'] = '#d9d9d9'
plt.rcParams['xtick.color'] = '#aaa'
plt.rcParams['ytick.color'] = '#aaa'
plt.rcParams['text.color'] = '#d9d9d9'
plt.rcParams['grid.color'] = '#1e1d6a'
ACCENT = '#6c63ff'
ACCENT2 = '#00c882'
DANGER = '#ff6b6b'

# Caminho dos dados — mesmo padrão do main_analysis.ipynb
BASE_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
PROCESSED = os.path.join(BASE_DIR, 'data', 'processed', 'olist_super_dataset.csv')
RAW_DIR   = os.path.join(BASE_DIR, 'data', 'raw')

print('Carregando Super Dataset...')
df = pd.read_csv(PROCESSED, low_memory=False)
print(f'Dataset carregado: {df.shape[0]:,} linhas x {df.shape[1]} colunas')
df.head(3)

## Fase 2: Feature Engineering

Preparamos as variáveis que os modelos vão usar. Seguimos o mesmo `feature engineering` do `utils.py` do Streamlit para garantir consistência entre o notebook de pesquisa e o dashboard em produção.

In [ ]:
# Conversão de datas
date_cols = [
    'order_purchase_timestamp',
    'order_estimated_delivery_date',
    'order_delivered_customer_date',
    'order_approved_at'
]
for col in date_cols:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], errors='coerce')

# --- Features Temporais ---
if 'order_purchase_timestamp' in df.columns:
    df['mes_compra']        = df['order_purchase_timestamp'].dt.month
    df['dia_semana_compra'] = df['order_purchase_timestamp'].dt.dayofweek  # 0=seg, 6=dom
    df['hora_compra']       = df['order_purchase_timestamp'].dt.hour
    df['fim_de_semana']     = (df['dia_semana_compra'] >= 5).astype(int)

# --- Target: Atraso ---
if 'order_delivered_customer_date' in df.columns and 'order_estimated_delivery_date' in df.columns:
    df['dias_atraso']  = (df['order_delivered_customer_date'] - df['order_estimated_delivery_date']).dt.days
    df['flag_atraso']  = (df['dias_atraso'] > 0).astype(int)

# --- Target: Tempo de Entrega Real (dias) ---
if 'order_delivered_customer_date' in df.columns and 'order_approved_at' in df.columns:
    df['tempo_entrega_real'] = (df['order_delivered_customer_date'] - df['order_approved_at']).dt.days

# --- Feature: Rota interestadual ---
if 'customer_state' in df.columns and 'seller_state' in df.columns:
    df['rota_interestadual'] = (df['customer_state'] != df['seller_state']).astype(int)

# --- Feature: Região do cliente ---
REGIAO_MAP = {
    'AC':'Norte','AP':'Norte','AM':'Norte','PA':'Norte','RO':'Norte','RR':'Norte','TO':'Norte',
    'AL':'Nordeste','BA':'Nordeste','CE':'Nordeste','MA':'Nordeste','PB':'Nordeste',
    'PE':'Nordeste','PI':'Nordeste','RN':'Nordeste','SE':'Nordeste',
    'DF':'Centro-Oeste','GO':'Centro-Oeste','MT':'Centro-Oeste','MS':'Centro-Oeste',
    'ES':'Sudeste','MG':'Sudeste','RJ':'Sudeste','SP':'Sudeste',
    'PR':'Sul','RS':'Sul','SC':'Sul'
}
if 'customer_state' in df.columns:
    df['regiao_cliente'] = df['customer_state'].map(REGIAO_MAP).fillna('Outros')

print('Feature engineering concluído.')
print(f"Taxa de atraso na base: {df['flag_atraso'].mean()*100:.1f}%")
print(f"Tempo médio de entrega: {df['tempo_entrega_real'].median():.0f} dias (mediana)")
df[['flag_atraso','tempo_entrega_real','rota_interestadual','regiao_cliente']].describe()

## Fase 3: Análise Exploratória Focada em ML

Antes de treinar qualquer modelo, precisamos entender a distribuição dos targets e confirmar quais features têm poder preditivo real.

In [ ]:
# Distribuição do target de classificação
if 'flag_atraso' in df.columns:
    contagem = df['flag_atraso'].value_counts()
    labels = ['No Prazo', 'Atrasado']
    cores  = [ACCENT2, DANGER]

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    # Pizza
    axes[0].pie(contagem, labels=labels, colors=cores, autopct='%1.1f%%',
                startangle=90, textprops={'color': '#d9d9d9', 'fontsize': 12})
    axes[0].set_title('Distribuição: No Prazo vs Atrasado', color='#d9d9d9', fontsize=13)

    # Nota por atraso
    if 'review_score' in df.columns:
        medias = df.groupby('flag_atraso')['review_score'].mean()
        bars = axes[1].bar(['No Prazo', 'Atrasado'], medias.values, color=cores, width=0.5, edgecolor='none')
        axes[1].set_ylim(0, 5)
        axes[1].set_ylabel('Review Score Médio', color='#d9d9d9')
        axes[1].set_title('Impacto do Atraso na Avaliação', color='#d9d9d9', fontsize=13)
        for bar, val in zip(bars, medias.values):
            axes[1].text(bar.get_x() + bar.get_width()/2, val + 0.05, f'{val:.2f}',
                         ha='center', color='#d9d9d9', fontsize=13, fontweight='bold')

    plt.tight_layout()
    plt.show()
    print(f"\nPedidos no prazo:  {contagem.get(0,0):,} ({contagem.get(0,0)/len(df)*100:.1f}%)")
    print(f"Pedidos atrasados: {contagem.get(1,0):,} ({contagem.get(1,0)/len(df)*100:.1f}%)")

In [ ]:
# Taxa de atraso por região — confirma que região é preditor forte
if 'regiao_cliente' in df.columns and 'flag_atraso' in df.columns:
    atraso_regiao = (
        df.groupby('regiao_cliente')['flag_atraso']
        .mean()
        .sort_values(ascending=False)
        .reset_index()
    )
    atraso_regiao['pct'] = atraso_regiao['flag_atraso'] * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    cores_barra = [DANGER if v > atraso_regiao['pct'].mean() else ACCENT for v in atraso_regiao['pct']]
    bars = ax.barh(atraso_regiao['regiao_cliente'], atraso_regiao['pct'],
                   color=cores_barra, edgecolor='none')
    ax.axvline(atraso_regiao['pct'].mean(), color='#ffd93d', linestyle='--',
               linewidth=1.5, label=f"Média: {atraso_regiao['pct'].mean():.1f}%")
    ax.set_xlabel('Taxa de Atraso (%)', color='#d9d9d9')
    ax.set_title('Taxa de Atraso por Região — Confirmando Poder Preditivo', color='#d9d9d9', fontsize=13)
    ax.legend()
    for bar, val in zip(bars, atraso_regiao['pct']):
        ax.text(val + 0.3, bar.get_y() + bar.get_height()/2,
                f'{val:.1f}%', va='center', color='#d9d9d9', fontsize=10)
    plt.tight_layout()
    plt.show()

In [ ]:
# Correlação das features numéricas com o target
feature_candidates = [
    'freight_value', 'price', 'mes_compra', 'dia_semana_compra',
    'fim_de_semana', 'rota_interestadual', 'tempo_entrega_real'
]
available = [c for c in feature_candidates if c in df.columns] + ['flag_atraso']

corr = df[available].corr()
fig, ax = plt.subplots(figsize=(10, 6))
mask = np.zeros_like(corr, dtype=bool)
mask[np.triu_indices_from(mask)] = True
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, ax=ax, linewidths=0.5,
            cbar_kws={'shrink': 0.8})
ax.set_title('Correlação entre Features e Target (flag_atraso)', color='#d9d9d9', fontsize=13)
plt.tight_layout()
plt.show()

## Fase 4: Modelo 1 — Classificador de Atraso

**Objetivo:** Prever se um pedido vai atrasar (flag_atraso = 1) ou não (flag_atraso = 0).

**Por que Random Forest?**  
- Lida bem com features de tipos mistos (numérico + categórico codificado)  
- Robusto a outliers — comum em dados de frete e preço  
- Fornece `feature_importances_` nativamente, facilitando interpretação  
- Resultados sólidos sem necessidade de tunagem extensiva para um baseline

**Métrica principal: F1-Score** com foco em Recall — preferimos errar avisando um atraso que não vai acontecer (falso positivo) a deixar passar um atraso real sem avisar (falso negativo).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    classification_report, confusion_matrix,
    f1_score, roc_auc_score, ConfusionMatrixDisplay
)

# --- Seleção de Features ---
FEATURES_CLF = [
    'freight_value', 'price',
    'mes_compra', 'dia_semana_compra', 'fim_de_semana',
    'rota_interestadual',
    'customer_state', 'seller_state'
]
TARGET_CLF = 'flag_atraso'

cols_needed = FEATURES_CLF + [TARGET_CLF]
available_cols = [c for c in cols_needed if c in df.columns]
df_clf = df[available_cols].dropna()

# Encoding de variáveis categóricas
le_customer = LabelEncoder()
le_seller   = LabelEncoder()
if 'customer_state' in df_clf.columns:
    df_clf = df_clf.copy()
    df_clf['customer_state_enc'] = le_customer.fit_transform(df_clf['customer_state'])
    df_clf['seller_state_enc']   = le_seller.fit_transform(df_clf['seller_state'])
    df_clf = df_clf.drop(columns=['customer_state', 'seller_state'])

X = df_clf.drop(columns=[TARGET_CLF])
y = df_clf[TARGET_CLF]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Treino: {len(X_train):,} amostras | Teste: {len(X_test):,} amostras')
print(f'Proporção de atraso no treino: {y_train.mean()*100:.1f}%')

In [ ]:
# Treinamento
clf = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=10,
    class_weight='balanced',  # Compensa o desbalanceamento (mais 'no prazo' que 'atrasado')
    random_state=42,
    n_jobs=-1
)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
y_prob = clf.predict_proba(X_test)[:, 1]

print('=== Resultados — Classificador de Atraso ===')
print(classification_report(y_test, y_pred, target_names=['No Prazo', 'Atrasado']))
print(f'F1-Score (Atrasado): {f1_score(y_test, y_pred):.4f}')
print(f'AUC-ROC:             {roc_auc_score(y_test, y_prob):.4f}')

In [ ]:
# Matriz de Confusão + Importância de Features
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Matriz de confusão
cm = confusion_matrix(y_test, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=['No Prazo', 'Atrasado'])
disp.plot(ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusão — Classificador de Atraso', color='#d9d9d9', fontsize=12)
axes[0].set_facecolor('#0d0c44')

# Feature Importance
feat_imp = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=True)
cores_imp = [ACCENT if v >= feat_imp.quantile(0.7) else '#3a3880' for v in feat_imp]
feat_imp.plot(kind='barh', ax=axes[1], color=cores_imp, edgecolor='none')
axes[1].set_title('Importância das Features', color='#d9d9d9', fontsize=12)
axes[1].set_xlabel('Importância Relativa', color='#d9d9d9')

plt.tight_layout()
plt.show()

## Fase 5: Modelo 2 — Predição de Prazo de Entrega (Regressão)

**Objetivo:** Prever o número de dias até a entrega para exibir uma estimativa realista ao cliente no momento da compra.

**Por que Random Forest Regressor?**  
- Captura relações não-lineares (ex: o impacto do frete não é linear — dobrar o peso não dobra o prazo)  
- As mesmas features do classificador se aproveitam, facilitando manutenção  
- Baseline sólido antes de partir para modelos mais complexos (XGBoost, LightGBM)

**Métrica principal: MAE (Erro Médio Absoluto)** — em dias. Mais interpretável que RMSE para este contexto: "o modelo erra em média X dias".

In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

FEATURES_REG = [
    'freight_value', 'price',
    'mes_compra', 'dia_semana_compra', 'fim_de_semana',
    'rota_interestadual',
    'customer_state', 'seller_state'
]
TARGET_REG = 'tempo_entrega_real'

cols_needed_reg = FEATURES_REG + [TARGET_REG]
available_reg = [c for c in cols_needed_reg if c in df.columns]
df_reg = df[available_reg].dropna()
df_reg = df_reg[(df_reg[TARGET_REG] > 0) & (df_reg[TARGET_REG] < 100)]  # Remove outliers extremos

# Encoding
df_reg = df_reg.copy()
if 'customer_state' in df_reg.columns:
    df_reg['customer_state_enc'] = le_customer.fit_transform(df_reg['customer_state'])
    df_reg['seller_state_enc']   = le_seller.fit_transform(df_reg['seller_state'])
    df_reg = df_reg.drop(columns=['customer_state', 'seller_state'])

X_r = df_reg.drop(columns=[TARGET_REG])
y_r = df_reg[TARGET_REG]

X_r_train, X_r_test, y_r_train, y_r_test = train_test_split(
    X_r, y_r, test_size=0.2, random_state=42
)

reg = RandomForestRegressor(
    n_estimators=200,
    max_depth=14,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1
)
reg.fit(X_r_train, y_r_train)

y_r_pred = reg.predict(X_r_test)
mae  = mean_absolute_error(y_r_test, y_r_pred)
rmse = mean_squared_error(y_r_test, y_r_pred) ** 0.5
r2   = r2_score(y_r_test, y_r_pred)

print('=== Resultados — Predição de Prazo de Entrega ===')
print(f'MAE  (erro médio em dias): {mae:.2f} dias')
print(f'RMSE:                      {rmse:.2f} dias')
print(f'R²:                        {r2:.4f}')
print(f'\nInterpretação: o modelo erra em média {mae:.1f} dias no prazo de entrega.')

In [ ]:
# Predito vs Real + Distribuição do Erro
sample_idx = np.random.choice(len(y_r_test), size=min(3000, len(y_r_test)), replace=False)
y_real_sample = np.array(y_r_test)[sample_idx]
y_pred_sample = y_r_pred[sample_idx]
erro_sample = y_pred_sample - y_real_sample

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Scatter: predito vs real
axes[0].scatter(y_real_sample, y_pred_sample, alpha=0.3, color=ACCENT, s=10)
lim = max(y_real_sample.max(), y_pred_sample.max())
axes[0].plot([0, lim], [0, lim], color='#ffd93d', linewidth=1.5, linestyle='--', label='Perfeito')
axes[0].set_xlabel('Prazo Real (dias)', color='#d9d9d9')
axes[0].set_ylabel('Prazo Previsto (dias)', color='#d9d9d9')
axes[0].set_title(f'Predito vs Real  |  MAE = {mae:.1f} dias', color='#d9d9d9', fontsize=12)
axes[0].legend()

# Distribuição do erro
axes[1].hist(erro_sample, bins=50, color=ACCENT, edgecolor='none', alpha=0.8)
axes[1].axvline(0, color='#ffd93d', linewidth=1.5, linestyle='--', label='Erro zero')
axes[1].axvline(erro_sample.mean(), color=DANGER, linewidth=1.5,
                linestyle='-', label=f'Média: {erro_sample.mean():.1f} dias')
axes[1].set_xlabel('Erro (dias preditos − dias reais)', color='#d9d9d9')
axes[1].set_ylabel('Frequência', color='#d9d9d9')
axes[1].set_title('Distribuição do Erro de Previsão', color='#d9d9d9', fontsize=12)
axes[1].legend()

plt.tight_layout()
plt.show()

## Fase 6: Interpretação — O Que os Modelos Aprenderam?

Esta seção traduz os resultados técnicos para linguagem de negócio — essencial para apresentar ao time e embasar a página 11 do Streamlit.

In [ ]:
# Comparação de importância: Classificador vs Regressor
imp_clf = pd.Series(clf.feature_importances_, index=X.columns).sort_values(ascending=False)
imp_reg = pd.Series(reg.feature_importances_, index=X_r.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, imp, title, cor in [
    (axes[0], imp_clf, 'Classificador de Atraso (flag_atraso)', DANGER),
    (axes[1], imp_reg, 'Regressor de Prazo (tempo_entrega_real)', ACCENT)
]:
    imp_top = imp.head(8).sort_values()
    ax.barh(imp_top.index, imp_top.values, color=cor, alpha=0.85, edgecolor='none')
    ax.set_title(title, color='#d9d9d9', fontsize=11)
    ax.set_xlabel('Importância Relativa', color='#d9d9d9')

plt.suptitle('O Que os Modelos Aprenderam — Features Mais Relevantes', color='#d9d9d9', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Prazo médio previsto por região — visualização para o Streamlit
if 'regiao_cliente' in df.columns:
    df_reg_plot = df[['regiao_cliente', 'tempo_entrega_real', 'flag_atraso']].dropna()
    resumo_regiao = df_reg_plot.groupby('regiao_cliente').agg(
        prazo_medio=('tempo_entrega_real', 'median'),
        taxa_atraso=('flag_atraso', 'mean')
    ).reset_index().sort_values('prazo_medio', ascending=False)
    resumo_regiao['taxa_atraso_pct'] = resumo_regiao['taxa_atraso'] * 100

    fig, ax = plt.subplots(figsize=(10, 4))
    cores_reg = [DANGER if t > resumo_regiao['taxa_atraso_pct'].mean() else ACCENT
                 for t in resumo_regiao['taxa_atraso_pct']]
    bars = ax.bar(resumo_regiao['regiao_cliente'], resumo_regiao['prazo_medio'],
                  color=cores_reg, edgecolor='none')
    ax.set_ylabel('Prazo Mediano (dias)', color='#d9d9d9')
    ax.set_title('Prazo Mediano de Entrega por Região', color='#d9d9d9', fontsize=13)
    for bar, row in zip(bars, resumo_regiao.itertuples()):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                f'{row.prazo_medio:.0f}d\n({row.taxa_atraso_pct:.0f}% atraso)',
                ha='center', va='bottom', color='#d9d9d9', fontsize=9)
    plt.tight_layout()
    plt.show()

    print(resumo_regiao.to_string(index=False))

## Fase 7: Resumo Executivo e Próximos Passos

### Resultados dos Modelos

| Modelo | Task | Métrica Principal | Interpretação |
|---|---|---|---|
| Random Forest Classifier | Predição de Atraso | F1-Score + AUC-ROC | Ver saída da Fase 4 |
| Random Forest Regressor | Prazo de Entrega | MAE em dias | Ver saída da Fase 5 |

### O Que Foi Confirmado

- **Rota interestadual** e **estado do cliente** são consistentemente as features mais importantes nos dois modelos
- **Valor do frete** é um proxy poderoso do risco logístico (fretes altos = produtos pesados = mais risco)
- **Dia da semana e mês** têm importância secundária mas mensurável — compras em sexta/fim de semana têm prazo maior

### Próximos Passos Técnicos

1. **Adicionar features de dimensões físicas do produto** (`product_weight_g`, `product_volume_cm3`) — provavelmente aumentarão o poder preditivo  
2. **Testar XGBoost / LightGBM** como upgrade do Random Forest — ganho de performance com tempo de treino similar  
3. **Calibração de probabilidade** no classificador para uso em produção (Platt Scaling)  
4. **Integrar com o Streamlit** — as previsões deste modelo alimentam a página `11_Machine_Learning.py`

### Conexão com o Dashboard

Os modelos treinados aqui podem ser serializados com `joblib` e carregados no Streamlit:

```python
import joblib
joblib.dump(clf, 'models/clf_atraso.pkl')
joblib.dump(reg, 'models/reg_prazo.pkl')
```

Na página 11, o usuário poderia simular: *"Se o cliente for do Norte e o produto pesar 5kg, qual a probabilidade de atraso?"*

In [ ]:
# Resumo final impresso
print('=' * 55)
print('  RESUMO DOS MODELOS — NOTEBOOK ML / LUÍS')
print('=' * 55)

try:
    print(f'\n[Classificador de Atraso]')
    print(f'  F1-Score : {f1_score(y_test, y_pred):.4f}')
    print(f'  AUC-ROC  : {roc_auc_score(y_test, y_prob):.4f}')
    print(f'  Features : {list(X.columns)}')
except:
    print('  (execute as células da Fase 4 primeiro)')

try:
    print(f'\n[Regressor de Prazo]')
    print(f'  MAE  : {mae:.2f} dias')
    print(f'  RMSE : {rmse:.2f} dias')
    print(f'  R²   : {r2:.4f}')
    print(f'  Features : {list(X_r.columns)}')
except:
    print('  (execute as células da Fase 5 primeiro)')

print('\n[Arquivo]')
print('  notebooks/Luis/ml_modelo_preditivo_classificacao.ipynb')
print('  Destino: src/pages/11_Machine_Learning.py (conteúdo)')